# Painel de Análise de Vendas

Atividade da apostila, organizada em células de código como os notebooks das aulas anteriores. Execute na pasta Apostila, de cima para baixo.


## Etapa 1 — Preparação e importação


In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
sns.set_palette("deep")
pd.options.display.float_format = "{:,.2f}".format
Path("graficos").mkdir(exist_ok=True)

In [ ]:
# Cria um CSV fictício reproduzível para a atividade.
rng = np.random.default_rng(42)
lojas = {
    "Centro": "Sudeste",
    "Norte": "Norte",
    "Sul": "Sul",
    "Leste": "Nordeste",
}
produtos = {
    "Camiseta": ("Vestuário", 49.90),
    "Calça": ("Vestuário", 119.90),
    "Tênis": ("Calçados", 199.90),
    "Sandália": ("Calçados", 89.90),
    "Mochila": ("Acessórios", 139.90),
    "Boné": ("Acessórios", 39.90),
}
linhas = []
for mes in range(1, 13):
    for loja, regiao in lojas.items():
        for produto, (categoria, preco) in produtos.items():
            for _ in range(3):
                dia = int(rng.integers(1, 29))
                quantidade = int(rng.integers(1, 13)) + (2 if mes in (11, 12) else 0)
                linhas.append([f"2025-{mes:02d}-{dia:02d}", loja, regiao, produto, categoria, quantidade, preco])

dados = pd.DataFrame(
    linhas,
    columns=["data", "loja", "regiao", "produto", "categoria", "quantidade", "preco_unitario"],
)
dados.to_csv("vendas.csv", index=False, encoding="utf-8-sig")
dados.head()

In [ ]:
vendas = pd.read_csv("vendas.csv", parse_dates=["data"])
vendas.head()

In [ ]:
vendas.info()
vendas.isnull().sum()

In [ ]:
vendas.describe(include="all")

In [ ]:
for coluna in ["quantidade", "preco_unitario"]:
    vendas[coluna] = pd.to_numeric(vendas[coluna], errors="coerce")
    vendas[coluna] = vendas[coluna].fillna(vendas[coluna].mean())
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas["mes"] = vendas["data"].dt.month
vendas["mes_ano"] = vendas["data"].dt.to_period("M")
display(vendas.isnull().sum().to_frame("valores_ausentes_apos_tratamento"))

## Etapa 2 — Análise exploratória

Cada linha representa uma venda.


In [ ]:
faturamento_loja = vendas.groupby("loja")["faturamento"].sum().sort_values(ascending=False)
ticket = vendas.groupby("loja")["faturamento"].agg(["sum", "count"])
ticket["ticket_medio"] = ticket["sum"] / ticket["count"]
top_produtos = vendas.groupby("produto")["quantidade"].sum().nlargest(5)
mensal = vendas.groupby("mes_ano")["faturamento"].sum().sort_index()
regiao = vendas.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)
display(faturamento_loja.to_frame("faturamento"))
display(ticket[["ticket_medio"]])
display(top_produtos.to_frame("quantidade"))
print("Maior mês:", mensal.idxmax().strftime("%m/%Y"), "R$", round(mensal.max(), 2))
print("Menor mês:", mensal.idxmin().strftime("%m/%Y"), "R$", round(mensal.min(), 2))

## Etapa 3 — Visualizações

Tema whitegrid e paleta deep aplicados aos gráficos.


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(faturamento_loja.index, faturamento_loja.values, color=sns.color_palette("deep", len(faturamento_loja)), label="Faturamento")
ax.bar_label(bars, fmt="R$ %.0f", padding=3)
ax.set(title="Faturamento total por loja", xlabel="Loja", ylabel="Faturamento (R$)")
ax.legend()
fig.tight_layout(); fig.savefig(Path("graficos")/"faturamento_por_loja.png",dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
labels = mensal.index.strftime("%m/%Y")
ax.plot(labels, mensal.values, marker="o", label="Faturamento mensal")
for x,y in zip(labels,mensal.values):
    ax.annotate(f"{y/1000:.1f} mil",(x,y),xytext=(0,8),textcoords="offset points",ha="center",fontsize=8)
ax.set(title="Evolução mensal do faturamento",xlabel="Mês/Ano",ylabel="Faturamento (R$)")
ax.tick_params(axis="x",rotation=45); ax.legend()
fig.tight_layout(); fig.savefig(Path("graficos")/"evolucao_mensal.png",dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ax.pie(regiao.values,labels=regiao.index,autopct="%1.1f%%",startangle=90,colors=sns.color_palette("deep",len(regiao)))
ax.set_title("Participação das regiões no faturamento")
fig.tight_layout(); fig.savefig(Path("graficos")/"faturamento_por_regiao.png",dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.scatterplot(data=vendas,x="quantidade",y="faturamento",hue="categoria",alpha=.65,ax=ax)
ax.set(title="Quantidade e faturamento por venda",xlabel="Quantidade de itens",ylabel="Faturamento (R$)")
ax.legend(title="Categoria")
fig.tight_layout(); fig.savefig(Path("graficos")/"dispersao_quantidade_faturamento.png",dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.boxplot(data=vendas,x="categoria",y="preco_unitario",hue="categoria",palette="deep",legend=False,ax=ax)
ax.set(title="Preços unitários por categoria",xlabel="Categoria",ylabel="Preço unitário (R$)")
fig.tight_layout(); fig.savefig(Path("graficos")/"precos_por_categoria.png",dpi=150); plt.show()

In [ ]:
corr = vendas[["quantidade","preco_unitario","faturamento","mes"]].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm",vmin=-1,vmax=1,ax=ax)
ax.set_title("Correlação entre variáveis numéricas")
fig.tight_layout(); fig.savefig(Path("graficos")/"correlacao.png",dpi=150); plt.show()

## Etapa 4 — Layout e relatório final


In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,5))
ax1.bar(faturamento_loja.index,faturamento_loja.values,color=sns.color_palette("deep",len(faturamento_loja)))
ax1.set(title="Faturamento por loja",xlabel="Loja",ylabel="Faturamento (R$)")
ax2.bar(regiao.index,regiao.values,color=sns.color_palette("deep",len(regiao)))
ax2.set(title="Faturamento por região",xlabel="Região",ylabel="Faturamento (R$)")
ax2.tick_params(axis="x",rotation=30)
fig.tight_layout(); fig.savefig(Path("graficos")/"comparativo_loja_regiao.png",dpi=150); plt.show()

In [ ]:
resumo = ticket.rename(columns={"sum":"faturamento_total","count":"numero_vendas"}).reset_index()
resumo["participacao_percentual"] = 100*resumo["faturamento_total"]/resumo["faturamento_total"].sum()
resumo = resumo.sort_values("faturamento_total",ascending=False)
resumo.to_csv(Path("vendas.csv").parent/"resumo_analitico.csv",index=False,encoding="utf-8-sig")
display(resumo)
print(f"Loja líder: {faturamento_loja.index[0]} (R$ {faturamento_loja.iloc[0]:,.2f}).")
print(f"Região líder: {regiao.index[0]} ({100*regiao.iloc[0]/regiao.sum():.1f}% do total).")
print(f"Maior mês: {mensal.idxmax().strftime('%m/%Y')}; menor mês: {mensal.idxmin().strftime('%m/%Y')}.")
print(f"Produto com mais unidades: {top_produtos.index[0]} ({top_produtos.iloc[0]}).")
print("A dispersão relaciona quantidade e faturamento; o boxplot compara a variação de preços entre categorias.")